# Memory

<div style="display: flex; justify-content: flex-start; gap: 10px;">
  <img src="./assets/LC_Memory_after.png" style="width:300px; border:1px solid #ccc; border-radius:6px;">
</div>

Persisting messages, or 'agent state' between invocations of the agent.

## Setup

Load and/or check for needed environmental variables

In [1]:
from dotenv import load_dotenv
from env_utils import doublecheck_env

# Load environment variables from .env
load_dotenv()

# Check and print results
doublecheck_env("example.env")

OPENAI_API_KEY=****here
LANGSMITH_API_KEY=****2126
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=****ials


In [2]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///Chinook.db")

In [3]:
from dataclasses import dataclass

@dataclass
class RuntimeContext:
    db: SQLDatabase

In [4]:
from langchain_core.tools import tool
from langgraph.runtime import get_runtime

@tool
def execute_sql(query: str) -> str:
    """Execute a SQLite command and return results."""
    runtime = get_runtime(RuntimeContext)
    db = runtime.context.db

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [5]:
# Fetch the schema dynamically so the LLM knows the table/column names
schema = db.get_table_info()

SYSTEM_PROMPT = f"""You are a careful SQLite analyst.

Here is the database schema:
{schema}

Rules:
- Think step-by-step. 
- You often need to JOIN 'Customer' and 'Invoice' tables to find data by name.
- When you need data, call the tool `execute_sql` with ONE SELECT query.
- Read-only only.
- Limit to 5 rows unless the user explicitly asks otherwise.
- If the tool returns 'Error:', revise the SQL and try again.
"""

In [6]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="qwen3:14b", temperature=0)

In [7]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
)

## Repeated Queries

In [8]:
question = "This is Frank Harris, What was the total on my last invoice?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values",
    context=RuntimeContext(db=db),
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

This is Frank Harris, What was the total on my last invoice?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (a281d980-3ced-4feb-9803-3a5d89ffd5aa)
 Call ID: a281d980-3ced-4feb-9803-3a5d89ffd5aa
  Args:
    query: SELECT CustomerId FROM Customer WHERE FirstName = 'Frank' AND LastName = 'Harris';
================================= Tool Message =================================
Name: execute_sql

[(16,)]
================================== Ai Message ==================================
Tool Calls:
  execute_sql (fd5aea25-6b0e-4ba4-84ac-8a5bb709e782)
 Call ID: fd5aea25-6b0e-4ba4-84ac-8a5bb709e782
  Args:
    query: SELECT Total FROM Invoice WHERE CustomerId = 16 ORDER BY InvoiceDate DESC LIMIT 1;
================================= Tool Message =================================
Name: execute_sql

[(5.94,)]
================================== Ai Message ===

In [9]:
question = "What were the titles?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What were the titles?
================================== Ai Message ==================================

The question "What were the titles?" is ambiguous because multiple tables contain 'Title' columns (e.g., `Album`, `Track`). Could you clarify which table's titles you're referring to? For example:
- Are you asking for **album titles**?
- Or **track titles**?
- Something else?

Let me know so I can provide the correct information.


## Add memory

In [10]:
from langgraph.checkpoint.memory import InMemorySaver

In [11]:
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage

agent = create_agent(
    model=llm,
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
    checkpointer=InMemorySaver(),
)

In [12]:
question = "This is Frank Harris, What was the total on my last invoice?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

This is Frank Harris, What was the total on my last invoice?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (139dbfd9-7b43-4782-ae07-62f45cbed504)
 Call ID: 139dbfd9-7b43-4782-ae07-62f45cbed504
  Args:
    query: SELECT CustomerId FROM Customer WHERE FirstName = 'Frank' AND LastName = 'Harris';
================================= Tool Message =================================
Name: execute_sql

[(16,)]
================================== Ai Message ==================================
Tool Calls:
  execute_sql (de7ced56-b3c6-4c4f-837e-70f8fd3336e1)
 Call ID: de7ced56-b3c6-4c4f-837e-70f8fd3336e1
  Args:
    query: SELECT Total FROM Invoice WHERE CustomerId = 16 ORDER BY InvoiceDate DESC LIMIT 1;
================================= Tool Message =================================
Name: execute_sql

[(5.94,)]
================================== Ai Message ===

In [13]:
question = "What were the titles?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

What were the titles?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (82c98ad4-39dd-4859-94bf-ddaa485c6b8e)
 Call ID: 82c98ad4-39dd-4859-94bf-ddaa485c6b8e
  Args:
    query: SELECT Track.Name FROM Track JOIN InvoiceLine ON Track.TrackId = InvoiceLine.TrackId WHERE InvoiceLine.InvoiceId = (SELECT InvoiceId FROM Invoice WHERE CustomerId = 16 ORDER BY InvoiceDate DESC LIMIT 1);
================================= Tool Message =================================
Name: execute_sql

[('Holier Than Thou',), ('Through The Never',), ('My Friend Of Misery',), ('The Wait',), ('Blitzkrieg',), ('So What',)]
================================== Ai Message ==================================

The titles on your last invoice were:  
1. Holier Than Thou  
2. Through The Never  
3. My Friend Of Misery  
4. The Wait  
5. Blitzkrieg  
6. So What  

Let me know if you need 

## Try your own queries
Now that there is memory, check the agents recall!

In [ ]:
question = "Print the first five rows of each table in the database"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

Print the first five rows of each table in the database
================================== Ai Message ==================================
Tool Calls:
  execute_sql (ebd4b5a9-1446-4748-a895-a8c2e9da0de3)
 Call ID: ebd4b5a9-1446-4748-a895-a8c2e9da0de3
  Args:
    query: SELECT * FROM Album LIMIT 5;
================================= Tool Message =================================
Name: execute_sql

[(1, 'For Those About To Rock We Salute You', 1), (2, 'Balls to the Wall', 2), (3, 'Restless and Wild', 2), (4, 'Let There Be Rock', 1), (5, 'Big Ones', 3)]
================================== Ai Message ==================================
Tool Calls:
  execute_sql (124868ef-a6fc-4d17-9094-b122c2e8b712)
 Call ID: 124868ef-a6fc-4d17-9094-b122c2e8b712
  Args:
    query: SELECT * FROM Artist LIMIT 5;
================================= Tool Message =================================
Name: execute_sql

[(1, 'AC/DC'), (2, 'Ac